# Lensless Camera Reconstruction Demo

This notebook demonstrates how to use the trained lensless reconstruction models.

## Setup

First, we'll clone the repository and install dependencies.

In [ ]:
# Clone the repository (replace with your actual repo URL)
!git clone https://YOUR_TOKEN@github.com/USERNAME/lenless.git

# Change to the repo directory
%cd lenless

In [ ]:
# Install dependencies
!pip install -r requirements.txt

## Download Checkpoints

Download the trained model checkpoints.

In [ ]:
# Download checkpoints
!python scripts/download_checkpoints.py

## Load Dataset

Provide a Google Drive URL to a zip file containing the dataset in the required format:
```
data_dir/
├── lensless/   *.png
├── masks/      *.npy
└── lensed/     *.png  (optional, ground truth)
```

In [ ]:
# User: provide your Google Drive zip URL here
DATASET_URL = "https://drive.google.com/file/d/YOUR_FILE_ID/view"  # Replace with actual URL

# Download and extract the dataset
import gdown
import zipfile
from pathlib import Path

# Download
zip_path = "/tmp/dataset.zip"
gdown.download(DATASET_URL, zip_path, fuzzy=True)

# Extract
extract_dir = "/tmp/dataset"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Dataset extracted to {extract_dir}")

## Run Inference

Run the model on the dataset to generate reconstructions.

In [ ]:
# Run inference
# This will save reconstructions to /tmp/reconstructions
!python inference.py \
    model=modular_pre_post \
    datasets=custom_dir \
    datasets.test.data_dir={extract_dir} \
    inferencer.save_path=reconstructions \
    inferencer.from_pretrained=saved/modular_pre_post.pth

## Visualize Results

Display some sample reconstructions.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import numpy as np

# Find sample images
lensless_dir = Path(extract_dir) / "lensless"
lensed_dir = Path(extract_dir) / "lensed"
recon_dir = Path("/tmp/reconstructions") / "test"

sample_files = sorted(lensless_dir.glob("*.png"))[:3]

fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes.reshape(1, -1)

for i, img_path in enumerate(sample_files):
    img_id = img_path.stem
    
    # Load images
    lensless = np.array(Image.open(img_path))
    recon = np.array(Image.open(recon_dir / f"{img_id}.png"))
    
    # Display
    axes[i, 0].imshow(lensless)
    axes[i, 0].set_title(f"Lensless ({img_id})")
    axes[i, 0].axis('off')
    
    if lensed_dir.exists():
        lensed = np.array(Image.open(lensed_dir / f"{img_id}.png"))
        axes[i, 1].imshow(lensed)
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis('off')
    else:
        axes[i, 1].text(0.5, 0.5, "No GT", ha='center', va='center')
        axes[i, 1].axis('off')
    
    axes[i, 2].imshow(recon)
    axes[i, 2].set_title("Reconstruction")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Calculate Metrics

If ground truth images are available, calculate reconstruction metrics.

In [ ]:
# Calculate metrics if ground truth exists
if lensed_dir.exists():
    !python calculate_metrics.py --gt_dir {lensed_dir} --pred_dir {recon_dir}
else:
    print("No ground truth available, skipping metrics calculation.")